# Append Sheets Across Files

Stacks several Excel workbooks that share the same sheet structure (e.g. one crawler
output per search keyword) into a single workbook: every sheet name found in any file is
appended across files, with a `source_file` column recording where each row came from.

> See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).

In [ ]:
import os
from datetime import datetime
import pandas as pd

# ---- Input: upload in Colab, or list local paths --------------------------------
try:
    from google.colab import files
    uploaded = files.upload()
    FILE_PATHS = list(uploaded.keys())
except ImportError:
    FILE_PATHS = ["../sample-data/amazon_keyword_a.xlsx", "../sample-data/amazon_keyword_b.xlsx"]

print(f"{len(FILE_PATHS)} files")

# ---- Collect every sheet name ---------------------------------------------------
all_sheets = set()
for path in FILE_PATHS:
    names = pd.ExcelFile(path).sheet_names
    print(f"{os.path.basename(path)}: {names}")
    all_sheets.update(names)

# ---- Append same-named sheets ---------------------------------------------------
merged = {}
for sheet in sorted(all_sheets):
    frames = []
    for path in FILE_PATHS:
        xls = pd.ExcelFile(path)
        if sheet not in xls.sheet_names:
            print(f"  - {os.path.basename(path)}: no '{sheet}' sheet, skipped")
            continue
        df = pd.read_excel(path, sheet_name=sheet).dropna(how="all")
        df.insert(0, "source_file", os.path.basename(path))
        frames.append(df)
    if frames:
        merged[sheet] = pd.concat(frames, ignore_index=True, sort=False)
        print(f"[{sheet}] {len(merged[sheet]):,} rows x {merged[sheet].shape[1]} cols")

# ---- Save -----------------------------------------------------------------------
out = f"appended_{datetime.now():%Y%m%d_%H%M%S}.xlsx"
with pd.ExcelWriter(out, engine="openpyxl") as w:
    for sheet, df in merged.items():
        df.to_excel(w, sheet_name=sheet[:31], index=False)   # Excel sheet-name limit
print("saved:", out)